# EDA · Ampliación R4 — Autocorrelación y estabilidad de clases

**Tarea de Jira:** DSAP-33 · **Dataset:** NYC 311 (`nyc311_diario_borough_tipo.parquet`)

**Objetivo:** Completar la exploración con análisis de autocorrelación (ACF/PACF) para identificar lags predictores y verificar la estabilidad de los umbrales de clasificación (bajo/medio/alto) entre años.

Este notebook parte de la serie semanal ya construida en `eda_calidad_unificado.ipynb`.

## 0. Configuración

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})

# Detecta la raíz del repo
REPO = Path(".").resolve()
while not (REPO / ".git").exists() and REPO != REPO.parent:
    REPO = REPO.parent

RUTA = REPO / "data" / "raw" / "nyc311_diario_borough_tipo.parquet"
print("Datos:", RUTA)

## 1. Reconstruir serie semanal por borough

Replica la misma lógica de `eda_calidad_unificado` para tener la serie base.

In [ ]:
df = pd.read_parquet(RUTA)
df["dia"] = pd.to_datetime(df["dia"])
print("Forma:", df.shape)
print("Rango:", df["dia"].min().date(), "->", df["dia"].max().date())
print("Boroughs:", df["borough"].unique())

In [ ]:
# Excluir Unspecified y agregar por semana
d = df[df["borough"] != "Unspecified"].copy()
d["semana"] = d["dia"].dt.to_period("W-SUN").dt.start_time

# Solo semanas completas
ini, fin = df["dia"].min(), df["dia"].max()
semanas = d["semana"].drop_duplicates().sort_values()
completas = semanas[(semanas >= ini) & (semanas + pd.Timedelta(days=6) <= fin)]

sem = (
    d[d["semana"].isin(completas)]
    .groupby(["semana", "borough"], as_index=False)["n"]
    .sum()
    .sort_values(["borough", "semana"])
)
sem["anio"] = sem["semana"].dt.year

print(f"Semanas completas: {sem['semana'].nunique()}")
print(f"Observaciones: {len(sem)} (semanas × boroughs)")
sem.head()

## 2. Autocorrelación (ACF) y Autocorrelación Parcial (PACF)

El ACF muestra la correlación de la serie con sus valores pasados. El PACF aísla la correlación directa de cada lag eliminando el efecto de los intermedios. Esto permite identificar qué rezagos semanales son predictores significativos para el modelo.

In [ ]:
boroughs = sorted(sem["borough"].unique())

fig, axes = plt.subplots(len(boroughs), 2, figsize=(14, 4 * len(boroughs)))

for i, borough in enumerate(boroughs):
    serie = sem[sem["borough"] == borough].set_index("semana")["n"].sort_index()
    
    plot_acf(serie, ax=axes[i, 0], lags=52, title=f"ACF — {borough}", zero=False)
    plot_pacf(serie, ax=axes[i, 1], lags=52, title=f"PACF — {borough}", zero=False, method="ywm")

plt.tight_layout()
plt.savefig(REPO / "reports" / "acf_pacf_boroughs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/acf_pacf_boroughs.png")

### 2.1 Lags significativos por borough

Extraemos numéricamente qué lags tienen correlación parcial significativa (fuera de la banda de confianza al 95%).

In [ ]:
from statsmodels.tsa.stattools import pacf

print("Lags con PACF significativo (|PACF| > 2/√N):")
print("=" * 50)

for borough in boroughs:
    serie = sem[sem["borough"] == borough].set_index("semana")["n"].sort_index()
    n = len(serie)
    umbral = 2 / np.sqrt(n)  # Banda de confianza al 95%
    
    valores_pacf = pacf(serie, nlags=52, method="ywm")
    significativos = [lag for lag in range(1, 53) if abs(valores_pacf[lag]) > umbral]
    
    print(f"\n{borough} (umbral = ±{umbral:.3f}):")
    for lag in significativos:
        print(f"  Lag {lag:2d}: PACF = {valores_pacf[lag]:+.3f}")

### 2.2 Interpretación

**Lag 1 es el predictor dominante en los 5 boroughs**, con PACF entre +0.588 (Bronx) y +0.833 (Queens). Esto confirma una fuerte **inercia semanal**: la demanda de la semana pasada es el mejor predictor individual de la demanda actual.

**Lags 2 y 3 son significativos en 4 de 5 boroughs** (Brooklyn, Manhattan, Queens, Staten Island), indicando que la demanda tiene memoria de 2–3 semanas. Queens muestra la estructura temporal más rica con 6 lags significativos (1, 2, 3, 8, 16, 27).

**Lag 8 aparece con signo negativo** en Bronx (−0.181), Manhattan (−0.229) y Queens (−0.263), sugiriendo un ciclo de corrección de ~2 meses donde un pico de demanda se compensa hacia abajo.

**No se detecta estacionalidad anual directa en PACF** — el lag 52 no es significativo en ningún borough. Aunque la serie tiene patrones estacionales (ya documentados en `eda_calidad_unificado`), el PACF indica que esa estacionalidad se captura indirectamente a través de los primeros lags y no requiere un rezago explícito de 52 semanas como feature.

**Resumen de lags significativos por borough:**

| Borough | Lags significativos | PACF lag 1 |
|---|---|---|
| Bronx | 1, 7, 8 | +0.588 |
| Brooklyn | 1, 2, 5, 45 | +0.616 |
| Manhattan | 1, 2, 8 | +0.651 |
| Queens | 1, 2, 3, 8, 16, 27 | +0.833 |
| Staten Island | 1, 2, 3, 22 | +0.635 |

**Recomendación para features del modelo:**
- **Incluir obligatoriamente:** lag 1, lag 2, lag 3 (cubren la inercia de corto plazo)
- **Incluir opcionalmente:** lag 7–8 (ciclo bisemanal/bimensual)
- **No incluir:** lag 52 (no significativo en PACF)
- **Complementar con:** media móvil de 4 semanas para suavizar ruido

## 3. Estabilidad de clases entre años

La variable objetivo clasifica cada semana-borough en nivel **bajo**, **medio** o **alto** según percentiles. Si los umbrales cambian mucho de un año a otro, el modelo tendría dificultad para generalizar.

In [ ]:
# Calcular percentiles 33 y 66 por borough y año
umbrales = (
    sem.groupby(["borough", "anio"])["n"]
    .quantile([0.33, 0.66])
    .unstack()
    .rename(columns={0.33: "p33 (bajo/medio)", 0.66: "p66 (medio/alto)"})
    .reset_index()
)

print("Umbrales por borough y año:")
print("=" * 60)
for borough in boroughs:
    sub = umbrales[umbrales["borough"] == borough]
    print(f"\n{borough}:")
    print(sub.to_string(index=False))

In [ ]:
# Visualización de los umbrales a lo largo del tiempo
fig, axes = plt.subplots(1, len(boroughs), figsize=(4 * len(boroughs), 4), sharey=False)

for i, borough in enumerate(boroughs):
    sub = umbrales[umbrales["borough"] == borough]
    ax = axes[i]
    ax.plot(sub["anio"], sub["p33 (bajo/medio)"], "o-", label="p33", color="#2196F3")
    ax.plot(sub["anio"], sub["p66 (medio/alto)"], "s-", label="p66", color="#F44336")
    ax.set_title(borough, fontsize=11)
    ax.set_xlabel("Año")
    ax.set_xticks(sub["anio"])
    if i == 0:
        ax.set_ylabel("Solicitudes semanales")
    ax.legend(fontsize=8)

plt.suptitle("Estabilidad de umbrales bajo/medio/alto por borough", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(REPO / "reports" / "estabilidad_clases.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/estabilidad_clases.png")

### 3.1 Coeficiente de variación de los umbrales

In [ ]:
# CV = desviación estándar / media — un CV bajo indica umbrales estables
cv = (
    umbrales.groupby("borough")[["p33 (bajo/medio)", "p66 (medio/alto)"]]
    .agg(lambda x: x.std() / x.mean() * 100)
    .round(1)
    .rename(columns=lambda c: f"CV% {c}")
)

print("Coeficiente de variación (%) de los umbrales entre años:")
print(cv)
print("\nCV < 10% → umbrales razonablemente estables")
print("CV > 15% → considerar umbrales adaptativos o ventana móvil")

### 3.2 Clasificación cruzada: etiqueta global vs etiqueta por año

Si los umbrales son estables, la clasificación con percentiles globales debería coincidir con la clasificación año a año.

In [ ]:
# Etiqueta con umbrales globales (todo el periodo)
def etiquetar(grupo):
    p33 = grupo["n"].quantile(0.33)
    p66 = grupo["n"].quantile(0.66)
    return pd.cut(grupo["n"], bins=[-np.inf, p33, p66, np.inf], labels=["bajo", "medio", "alto"])

sem["clase_global"] = sem.groupby("borough", group_keys=False).apply(
    lambda g: etiquetar(g)
)

# Etiqueta con umbrales por año
sem["clase_anual"] = sem.groupby(["borough", "anio"], group_keys=False).apply(
    lambda g: etiquetar(g)
)

# Concordancia
coinciden = (sem["clase_global"] == sem["clase_anual"]).mean() * 100
print(f"Concordancia global vs anual: {coinciden:.1f}%")
print()

# Tabla cruzada por borough
for borough in boroughs:
    sub = sem[sem["borough"] == borough]
    conc = (sub["clase_global"] == sub["clase_anual"]).mean() * 100
    print(f"{borough}: {conc:.1f}% de concordancia")
    ct = pd.crosstab(sub["clase_global"], sub["clase_anual"], margins=True)
    print(ct)
    print()

### 3.3 Interpretación

**Los umbrales son razonablemente estables entre años.** Todos los coeficientes de variación están por debajo del 10%, lo que indica que los percentiles p33 y p66 no cambian drásticamente de un año a otro:

| Borough | CV% p33 | CV% p66 |
|---|---|---|
| Bronx | 5.4% | 8.2% |
| Brooklyn | 7.2% | 6.3% |
| Manhattan | 5.2% | 6.9% |
| Queens | 5.9% | 9.2% |
| Staten Island | 6.2% | 4.4% |

Esto valida el uso de **percentiles globales** (calculados sobre los 4 años) como estrategia de etiquetado para la variable objetivo.

**Sin embargo, la concordancia global vs anual es moderada (67.3%)**, lo que indica que aunque los umbrales no varían mucho en magnitud absoluta, hay suficiente variación interanual para que ~33% de las semanas cambien de categoría al comparar etiquetas globales con etiquetas por año.

**Análisis por borough:**
- **Staten Island** es el más estable (77.4% de concordancia) — borough más pequeño con demanda más predecible.
- **Brooklyn** es el menos estable (60.6%) — se observa una tendencia alcista sostenida donde p33 sube de 17,055 (2022) a 20,226 (2025), lo que desplaza las etiquetas.
- **Queens** también muestra baja concordancia (62.5%) — misma tendencia alcista (p33 de 13,659 a 15,732).

**Conclusión:** Los umbrales globales son una estrategia válida (CV < 10%), pero el modelo debe incorporar la **tendencia temporal** (año o número de semana absoluto) como feature para compensar el drift observado en Brooklyn y Queens. Para entregas futuras, una alternativa sería recalcular umbrales con ventana móvil de 2 años.

## 4. Resumen de hallazgos para el reporte

### 4.1 Lags predictores identificados

El análisis de autocorrelación parcial (PACF) sobre 208 semanas (2022–2025) con umbral de significancia al 95% (±0.139) revela que:

- **Lag 1** (semana anterior) es significativo en los 5 boroughs, con PACF entre +0.588 y +0.833. Es el predictor más fuerte.
- **Lags 2–3** son significativos en 4 de 5 boroughs, indicando memoria de corto plazo de 2–3 semanas.
- **Lag 8** aparece con signo negativo en 3 boroughs, sugiriendo un ciclo de corrección bimensual.
- **Lag 52** (estacionalidad anual) **no es significativo** en ningún borough — la estacionalidad se captura indirectamente.

### 4.2 Estabilidad de clases

- Los umbrales p33/p66 tienen **CV < 10%** en todos los boroughs → los percentiles globales son estables y usables.
- La concordancia entre etiquetas globales y anuales es del **67.3%**, con Brooklyn (60.6%) y Queens (62.5%) mostrando mayor inestabilidad debido a una tendencia alcista sostenida en la demanda.

### 4.3 Recomendaciones para el modelo

1. **Features temporales a incluir:** lag 1, lag 2, lag 3, media móvil de 4 semanas, y opcionalmente lag 7–8.
2. **Estrategia de etiquetado:** usar percentiles globales (p33/p66 sobre los 4 años), complementando con año o semana absoluta como feature para capturar tendencia.
3. **No incluir:** lag 52 como feature (no significativo en PACF).
4. **Para futuras iteraciones:** evaluar umbrales adaptativos con ventana móvil de 2 años para mejorar concordancia en Brooklyn y Queens.